# Neural closed-loop identification of glucose-insulin dynamics

Identification of a patient-specific glucose model from artificial pancreas (PID) data.
The model is a DualSSM, `y = SSM_0(meal) - filter(SSM_1(insulin))`, initialized from a baseline
trained on the virtual patient 101 and fine-tuned on the target patient with one of two strategies:

- **Strategy 1 (s1)**: open-loop fine-tuning on the recorded meal and insulin signals.
- **Strategy 2 (s2)**: closed-loop fine-tuning, with the model in negative feedback with a differentiable model of the PID controller.

The notebook is executed cell by cell (by tag) by `run_identification.py`.

## Parameters

In [ ]:
# Overridden by papermill
patient = 1
strategy = "s1"         # "s1" or "s2"
exp_name = "s1"         # name of the experiment folder under models/ and results/

## Imports

In [ ]:
import copy
import json
import sys
from pathlib import Path

import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

sys.path.insert(0, str(Path.cwd() / "src"))

from config import get_config
from dataset import FakeScenarioDataset, LoadData, SimpleLoadData, load_impulse_test
from dual_ssm import DualSSM, build_deep_ssm
from models import ClosedLoopSystem_SSM, NonLinearController, monotonic_gain_loss
from parameters import Parameter, PID_functions
from utils import (ROOT, FIT_formula, LowPassFilter, TrainingDivergence, check_divergence, ensure_3d,
                   fun_start_controller, impulse_response_metrics, plot_glucose_insulin, save_metrics,
                   second_difference_l1, set_seed)
import neural_ssm_fastloop

neural_ssm_fastloop.apply()
torch.set_default_dtype(torch.float32)
SEED = 42
set_seed(SEED)

## Configuration and paths

In [ ]:
config = get_config()
SSM_param = config["SSM_param"]
learning_rate = config["learning_rate"]
cumulative_window, horizon = config["cumulative_window"], config["horizon"]

TS = 5                      # sampling time [min]
NUM_DAYS = 30
WARMUP_EPOCHS = 10          # closed-loop epochs trained on a shortened horizon
WARMUP_STEPS = 250
MAX_RESTARTS = 100          # closed-loop restarts with a new seed after a divergence
G_BAR = torch.tensor([110.0])

data_path = ROOT / "data" / "PID" / "train" / "sc_30days_identification"
baseline_data_path = ROOT / "data" / "train_101"
impulse_insulin_path = ROOT / "data" / "test_ir" / "sc_1_day_test_IR_insulin_30g_13h"
impulse_meal_path = ROOT / "data" / "test_ir" / "sc_1_day_test_IR_meal_80g_13h"

baseline_folder = ROOT / "models" / "baseline" / f"paz_{patient:03d}"
model_folder = ROOT / "models" / exp_name / f"paz_{patient:03d}"
results_folder = ROOT / "results" / exp_name
model_folder.mkdir(parents=True, exist_ok=True)
with open(ROOT / "models" / exp_name / "config.json", "w") as f:
    json.dump(config, f, indent=2)

mse = nn.MSELoss()
print(f"patient {patient}, strategy {strategy}, experiment {exp_name}")

## Closed-loop data

30 days of PID closed-loop data: the first 5 hours are discarded, then 80% training, 10% validation, 10% test (consecutive days).

In [ ]:
train_size = int(NUM_DAYS * 0.8 * 1440 // TS)
val_size = int(NUM_DAYS * 0.1 * 1440 // TS)
test_size = int(NUM_DAYS * 0.1 * 1440 // TS)
start_valid = 5 * 12

dataset = LoadData(patient, data_path, train_size + start_valid,
                   disable_booster_d_pos=config.get("disable_booster_d_pos", True))
scaler_glucose, scaler_insulin, scaler_meal = dataset.scaler_glucose, dataset.scaler_insulin, dataset.scaler_meal

train_dataset = Subset(dataset, range(start_valid, train_size + start_valid))
val_dataset = Subset(dataset, range(train_size + start_valid, train_size + start_valid + val_size))
test_dataset = Subset(dataset, range(train_size + start_valid + val_size, train_size + val_size + test_size))
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

# The training set is split into consecutive 48-hour sequences, processed as a single batch.
steps_per_batch = 48 * 12


def create_fixed_batches(total_length, chunk_size, start_idx=0):
    """Consecutive index blocks of length chunk_size; an incomplete last block is discarded."""
    batches, start = [], start_idx
    while start + chunk_size <= total_length:
        batches.append(list(range(start, start + chunk_size)))
        start += chunk_size
    return batches


train_indices = create_fixed_batches(len(train_dataset), steps_per_batch)


def make_sequence_loader(subset):
    """Loader returning all 48-hour training sequences of `subset` as one batch of (meal, insulin, glucose, time)."""
    def collate(batch):
        return tuple(torch.stack([subset[i][k] for i in batch]) for k in range(4))
    return DataLoader(train_indices, batch_size=len(train_indices), shuffle=False, collate_fn=collate, drop_last=True)


train_loader = make_sequence_loader(train_dataset)

sio.savemat(model_folder / "scalers.mat", {
    "glucose_low": scaler_glucose.params["low"].item(), "glucose_high": scaler_glucose.params["high"].item(),
    "insulin_low": scaler_insulin.params["low"].item(), "insulin_high": scaler_insulin.params["high"].item(),
    "meal_low": scaler_meal.params["low"].item(), "meal_high": scaler_meal.params["high"].item(),
})

loaded_parameters = Parameter(patient, data_path)
CF = loaded_parameters.Patient_par.CF

## Helpers

In [ ]:
def unpack(batch):
    return tuple(ensure_3d(x) for x in batch)


def build_closed_loop(Dual_SSM):
    controller = NonLinearController(loaded_parameters, PID_functions, dataset.basal_vec, scaler_glucose, scaler_insulin,
                                     disable_booster_d_pos=config.get("disable_booster_d_pos", True))
    return ClosedLoopSystem_SSM(Dual_SSM, controller)


def simulate_closed_loop(closed_loop, u0, u1, y, t, n_steps=None, zero_meal=False):
    """Closed-loop rollout from the first sample of each sequence, optionally truncated to n_steps.

    The controller state is initialized from the recorded data of the target patient.
    """
    saturation_error_init, glucose_PID_init = fun_start_controller(t, loaded_parameters, scaler_glucose, scaler_insulin, dataset)
    closed_loop.reset(saturation_error_init, glucose_PID_init, u1[:, :1, :])
    u = torch.cat((u0, u1), dim=2)[:, :n_steps, :]
    if zero_meal:
        u = u.clone()
        u[:, :, 0] = 0
    return closed_loop(u, t[:, :n_steps, :], y_t0=y[:, :1, :1], u1_t0=u1[:, :1, :])


def save_model(folder, state, train_losses, val_losses, epoch_best):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    torch.save(state, folder / "trained_models.pth")
    sio.savemat(folder / "training_log.mat", {"train_losses": train_losses, "val_losses": val_losses, "epoch_best": epoch_best})


def load_state(folder):
    return torch.load(Path(folder) / "trained_models.pth", map_location="cpu")

## Baseline: virtual patient 101

Open-loop data of the virtual patient 101, normalized with the scalers of the target patient:
`30days_101_M` is used to train the meal branch, `30days_101_I_v2` the insulin branch.

In [ ]:
baseline_train_size = int(NUM_DAYS * 0.8 * 1440 // TS)
baseline_val_size = int(NUM_DAYS * 0.1 * 1440 // TS)
baseline_val_range = range(baseline_train_size + start_valid, baseline_train_size + start_valid + baseline_val_size)

dataset_101_I = SimpleLoadData(baseline_data_path / "30days_101_I_v2", 101, scaler_insulin, scaler_glucose, scaler_meal)
dataset_101_M = SimpleLoadData(baseline_data_path / "30days_101_M", 101, scaler_insulin, scaler_glucose, scaler_meal)

train_dataset_101_I = Subset(dataset_101_I, range(start_valid, baseline_train_size + start_valid))
train_dataset_101_M = Subset(dataset_101_M, range(start_valid, baseline_train_size + start_valid))
val_loader_101_I = DataLoader(Subset(dataset_101_I, baseline_val_range), batch_size=len(baseline_val_range), shuffle=False)
val_loader_101_M = DataLoader(Subset(dataset_101_M, baseline_val_range), batch_size=len(baseline_val_range), shuffle=False)
train_loader_101_I = make_sequence_loader(train_dataset_101_I)
train_loader_101_M = make_sequence_loader(train_dataset_101_M)

### Baseline, open loop

Each branch is trained separately: SSM_1 on the insulin scenario (`y = -SSM_1(insulin)`), SSM_0 on the meal scenario (`y = SSM_0(meal)`).

In [ ]:
def train_single_branch(model, train_loader_b, val_loader_b, input_idx, sign, epochs):
    """Train one branch on one input channel and return the model with the lowest validation loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_losses, val_losses = [], []
    best_val, candidate, epoch_best = float("inf"), None, -1

    for epoch in range(epochs):
        model.train()
        loss_epoch = 0.0
        for batch in train_loader_b:
            u, y = ensure_3d(batch[input_idx]), ensure_3d(batch[2])
            optimizer.zero_grad()
            loss = mse(sign * model(u)[0], y)
            loss.backward()
            optimizer.step()
            loss_epoch += loss.item()
        train_losses.append(loss_epoch)

        model.eval()
        loss_val = 0.0
        with torch.no_grad():
            for batch in val_loader_b:
                u, y = ensure_3d(batch[input_idx]), ensure_3d(batch[2])
                loss_val += mse(sign * model(u)[0], y).item()
        val_losses.append(loss_val)

        if loss_val < best_val:
            best_val, epoch_best, candidate = loss_val, epoch, copy.deepcopy(model)
        if epoch % 100 == 0:
            print(f"epoch {epoch:5d} | train {loss_epoch:.6f} | val {loss_val:.6f}")

    return candidate, train_losses, val_losses, epoch_best


SSM_0_101 = build_deep_ssm(SSM_param, 0)
SSM_1_101 = build_deep_ssm(SSM_param, 1)
epochs = config["epochs_baseline_s1"]

SSM_1_101, train_losses, val_losses, epoch_best = train_single_branch(SSM_1_101, train_loader_101_I, val_loader_101_I, input_idx=1, sign=-1, epochs=epochs)
save_model(baseline_folder / "insulin_s1", {"state_dict": SSM_1_101.state_dict()}, train_losses, val_losses, epoch_best)

SSM_0_101, train_losses, val_losses, epoch_best = train_single_branch(SSM_0_101, train_loader_101_M, val_loader_101_M, input_idx=0, sign=1, epochs=epochs)
save_model(baseline_folder / "meal_s1", {"state_dict": SSM_0_101.state_dict()}, train_losses, val_losses, epoch_best)

### Baseline, closed loop

The meal branch is initialized with the open-loop baseline; a new insulin branch is trained, jointly with the meal branch,
in closed loop with the PID model on the insulin scenario of patient 101.

In [ ]:
meal_baseline = build_deep_ssm(SSM_param, 0)
meal_baseline.load_state_dict(load_state(baseline_folder / "meal_s1")["state_dict"], strict=False)

seed, restarts = SEED, 0
while True:
    try:
        set_seed(seed)
        SSM_1 = build_deep_ssm(SSM_param, 1)
        Dual_SSM = DualSSM(copy.deepcopy(meal_baseline), SSM_1, LowPassFilter())
        closed_loop = build_closed_loop(Dual_SSM)
        optimizer = torch.optim.Adam(Dual_SSM.parameters(), lr=learning_rate)

        train_losses, val_losses = [], []
        best_val, candidate, epoch_best = float("inf"), None, -1
        for epoch in range(config["epochs_baseline_s2"]):
            n_steps = WARMUP_STEPS if epoch < WARMUP_EPOCHS else None

            closed_loop.train()
            loss_epoch = 0.0
            for batch in train_loader_101_I:
                u0, u1, y, t = unpack(batch)
                optimizer.zero_grad()
                y_hat, _, _ = simulate_closed_loop(closed_loop, u0, u1, y, t, n_steps)
                check_divergence(y_hat)
                loss = mse(y_hat, y[:, :n_steps, :])
                loss.backward()
                optimizer.step()
                loss_epoch += loss.item()
            train_losses.append(loss_epoch)

            Dual_SSM.eval()
            loss_val = 0.0
            with torch.no_grad():
                for batch in val_loader_101_I:
                    u0, u1, y, t = unpack(batch)
                    y_hat, _, _ = simulate_closed_loop(closed_loop, u0, u1, y, t, n_steps)
                    check_divergence(y_hat)
                    loss_val += mse(y_hat, y[:, :n_steps, :]).item()
            val_losses.append(loss_val)

            if loss_val < best_val:
                best_val, epoch_best, candidate = loss_val, epoch, copy.deepcopy(Dual_SSM)
            print(f"epoch {epoch:5d} | train {loss_epoch:.6f} | val {loss_val:.6f}")
        break
    except TrainingDivergence:
        restarts += 1
        seed += 1
        if restarts >= MAX_RESTARTS:
            raise
        print(f"divergence detected, restart {restarts} with seed {seed}")

save_model(baseline_folder / "closed_loop_s2", {"state_dict": candidate.state_dict()}, train_losses, val_losses, epoch_best)

## Strategy 1: open-loop fine-tuning

Both branches are initialized with the open-loop baseline and fine-tuned on the closed-loop data of the patient, minimizing

`MSE + prior_weight_s1 * L_prior + L_der2 + L_der3`

where `L_prior` is the physiological prior on the insulin branch, `L_der2` penalizes the second differences of the branch
outputs and `L_der3` the third differences of the responses to synthetic worst-case impulses.

In [ ]:
SSM_0 = build_deep_ssm(SSM_param, 0)
SSM_1 = build_deep_ssm(SSM_param, 0)
SSM_0.load_state_dict(load_state(baseline_folder / "meal_s1")["state_dict"], strict=False)
SSM_1.load_state_dict(load_state(baseline_folder / "insulin_s1")["state_dict"], strict=False)
l_filter = LowPassFilter()
optimizer = torch.optim.Adam(list(SSM_0.parameters()) + list(SSM_1.parameters()), lr=learning_rate)
fake_scenario = FakeScenarioDataset(scaler_meal, scaler_insulin)


def s1_losses(u0, u1, y):
    y0_hat, _ = SSM_0(u0)
    y1_hat, _ = SSM_1(u1)
    y1_hat = l_filter(y1_hat)
    y_hat = y0_hat - y1_hat

    loss_fit = mse(y_hat, y)
    loss_prior = config["prior_weight_s1"] * monotonic_gain_loss(
        -y1_hat, u1, scaler_glucose, scaler_insulin, cumulative_window, horizon, CF=CF, G_bar=G_BAR, l_filter=l_filter)
    loss_der2 = config["loss_coeff_der"] * 0.5 * (second_difference_l1(y0_hat) + second_difference_l1(y1_hat))
    loss_der3 = fake_scenario.compute_der3_loss(SSM_0, SSM_1, l_filter, coeff=config["loss_coeff_der3"])
    return loss_fit, loss_prior, loss_der2, loss_der3


train_losses, val_losses = [], []
best_val, epoch_best = float("inf"), -1
for epoch in range(config["epochs_s1"]):
    SSM_0.train(); SSM_1.train()
    loss_epoch = np.zeros(4)
    for batch in train_loader:
        u0, u1, y, _ = unpack(batch)
        optimizer.zero_grad()
        losses = s1_losses(u0, u1, y)
        total = sum(losses)
        total.backward()
        optimizer.step()
        loss_epoch += [l.item() for l in losses]

    SSM_0.eval(); SSM_1.eval()
    loss_val = np.zeros(4)
    with torch.no_grad():
        for batch in val_loader:
            u0, u1, y, _ = unpack(batch)
            loss_val += [l.item() for l in s1_losses(u0, u1, y)]

    train_losses.append(loss_epoch.sum())
    val_losses.append(loss_val.sum())
    if loss_val.sum() < best_val:
        best_val, epoch_best = loss_val.sum(), epoch
        candidate_SSM_0, candidate_SSM_1 = copy.deepcopy(SSM_0), copy.deepcopy(SSM_1)
    if epoch % 100 == 0:
        print(f"epoch {epoch:5d} | train {loss_epoch.sum():.6f} (fit, prior, der2, der3 = {np.round(loss_epoch, 6)}) | "
              f"val {loss_val.sum():.6f} (fit, prior, der2, der3 = {np.round(loss_val, 6)})")

save_model(model_folder / "strategy_1",
           {"SSM_0_state_dict": candidate_SSM_0.state_dict(), "SSM_1_state_dict": candidate_SSM_1.state_dict()},
           train_losses, val_losses, epoch_best)

### Strategy 1: evaluation

FIT on the training sequences (median) and on the test days; peak glucose deviation and time to peak for an isolated insulin bolus and an isolated meal.

In [ ]:
SSM_0 = build_deep_ssm(SSM_param, 0)
SSM_1 = build_deep_ssm(SSM_param, 0)
state = load_state(model_folder / "strategy_1")
SSM_0.load_state_dict(state["SSM_0_state_dict"], strict=False)
SSM_1.load_state_dict(state["SSM_1_state_dict"], strict=False)
SSM_0.eval(); SSM_1.eval()
l_filter = LowPassFilter()


def predict_s1(u0, u1):
    y0_hat, _ = SSM_0(u0)
    y1_hat, _ = SSM_1(u1)
    return y0_hat - l_filter(y1_hat)


def denorm(x):
    return scaler_glucose.denormalize(x.detach()).numpy()


metrics = {"patient": patient}
with torch.no_grad():
    u0, u1, y, t = unpack(next(iter(train_loader)))
    y_np, y_hat_np = denorm(y), denorm(predict_s1(u0, u1))
    metrics["FIT_train"] = float(np.median([FIT_formula(y_np[i, :, 0], y_hat_np[i, :, 0]) for i in range(y_np.shape[0])]))
    plot_glucose_insulin(t[0, :, 0].numpy(), meal=u0[0, :, 0].numpy(), insulin=u1[0, :, 0].numpy(),
                         glucose=y_np[0, :, 0], predicted_glucose=y_hat_np[0, :, 0], title="Training set, first sequence")

    u0, u1, y, t = unpack(next(iter(test_loader)))
    y_np, y_hat_np = denorm(y), denorm(predict_s1(u0, u1))
    metrics["FIT_test"] = float(FIT_formula(y_np[0, :, 0], y_hat_np[0, :, 0]))
    plot_glucose_insulin(t[0, :, 0].numpy(), meal=u0[0, :, 0].numpy(), insulin=u1[0, :, 0].numpy(),
                         glucose=y_np[0, :, 0], predicted_glucose=y_hat_np[0, :, 0], title="Test set")

    for mode, path in (("insulin", impulse_insulin_path), ("meal", impulse_meal_path)):
        u0, u1, y, t = load_impulse_test(path, patient, scaler_meal, scaler_insulin, scaler_glucose, glucose_key="G")
        y_hat = predict_s1(u0, u1)
        metrics[f"delta_G_{mode}"], metrics[f"delta_t_{mode}"] = impulse_response_metrics(
            y_hat, u1 if mode == "insulin" else u0, scaler_glucose, mode)
        plot_glucose_insulin(t[0, :, 0].numpy(), meal=u0[0, :, 0].numpy(), insulin=u1[0, :, 0].numpy(),
                             glucose=denorm(y)[0, :, 0], predicted_glucose=denorm(y_hat)[0, :, 0], title=f"Impulse test, {mode}")

save_metrics(results_folder / f"paz_{patient:03d}.csv", metrics)
print(metrics)

## Strategy 2: closed-loop fine-tuning

The DualSSM is initialized with the closed-loop baseline and fine-tuned in negative feedback with the PID model.
With `correct_prior_for_s2`, the prior is evaluated on a second rollout with the meal input set to zero, after removing the
final value of the meal branch.

In [ ]:
fake_scenario = FakeScenarioDataset(scaler_meal, scaler_insulin)


def s2_losses(u0, u1, y, t, n_steps):
    y_hat, y0_hat, y1_hat = simulate_closed_loop(closed_loop, u0, u1, y, t, n_steps)
    check_divergence(y_hat)
    loss_fit = mse(y_hat, y[:, :n_steps, :])

    if config["correct_prior_for_s2"]:
        y_hat_nm, y0_hat_nm, _ = simulate_closed_loop(closed_loop, u0, u1, y, t, n_steps, zero_meal=True)
        check_divergence(y_hat_nm)
        y_prior = y_hat_nm[:, 1:, :] - y0_hat_nm[:, -1:, :]
    else:
        y_prior = -y1_hat[:, 1:, :]

    loss_prior = config["prior_weight_s2"] * monotonic_gain_loss(
        y_prior, u1[:, 1:n_steps, :], scaler_glucose, scaler_insulin, cumulative_window, horizon, CF=CF, G_bar=G_BAR,
        aggressive_coeff=config["aggressive_coeff_s2"], l_filter=l_filter)
    loss_der2 = config["loss_coeff_der"] * 0.5 * (second_difference_l1(y0_hat) + second_difference_l1(y1_hat))
    loss_der3 = fake_scenario.compute_der3_loss_s2(Dual_SSM, coeff=config["loss_coeff_der3"])
    return loss_fit, loss_prior, loss_der2, loss_der3


seed, restarts = SEED, 0
while True:
    try:
        set_seed(seed)
        l_filter = LowPassFilter()
        Dual_SSM = DualSSM(build_deep_ssm(SSM_param, 0), build_deep_ssm(SSM_param, 0), l_filter)
        Dual_SSM.load_state_dict(load_state(baseline_folder / "closed_loop_s2")["state_dict"], strict=False)
        closed_loop = build_closed_loop(Dual_SSM)
        optimizer = torch.optim.Adam(closed_loop.parameters(), lr=learning_rate)

        train_losses, val_losses = [], []
        best_val, candidate, epoch_best = float("inf"), None, -1
        for epoch in range(config["epochs_s2"]):
            n_steps = WARMUP_STEPS if epoch < WARMUP_EPOCHS else None

            closed_loop.train()
            loss_epoch = np.zeros(4)
            for batch in train_loader:
                u0, u1, y, t = unpack(batch)
                optimizer.zero_grad()
                losses = s2_losses(u0, u1, y, t, n_steps)
                total = sum(losses)
                total.backward()
                optimizer.step()
                loss_epoch += [l.item() for l in losses]

            Dual_SSM.eval()
            loss_val = np.zeros(4)
            with torch.no_grad():
                for batch in val_loader:
                    u0, u1, y, t = unpack(batch)
                    loss_val += [l.item() for l in s2_losses(u0, u1, y, t, n_steps)]

            train_losses.append(loss_epoch.sum())
            val_losses.append(loss_val.sum())
            if loss_val.sum() < best_val:
                best_val, epoch_best, candidate = loss_val.sum(), epoch, copy.deepcopy(Dual_SSM)
            print(f"epoch {epoch:5d} | train {loss_epoch.sum():.6f} (fit, prior, der2, der3 = {np.round(loss_epoch, 6)}) | "
                  f"val {loss_val.sum():.6f} (fit, prior, der2, der3 = {np.round(loss_val, 6)})")
        break
    except TrainingDivergence:
        restarts += 1
        seed += 1
        if restarts >= MAX_RESTARTS:
            raise
        print(f"divergence detected, restart {restarts} with seed {seed}")

save_model(model_folder / "strategy_2", {"Dual_SSM_state_dict": candidate.state_dict()}, train_losses, val_losses, epoch_best)

### Strategy 2: evaluation

Same metrics as strategy 1, computed on closed-loop rollouts of the identified model with the PID model.

In [ ]:
Dual_SSM = DualSSM(build_deep_ssm(SSM_param, 0), build_deep_ssm(SSM_param, 0), LowPassFilter())
Dual_SSM.load_state_dict(load_state(model_folder / "strategy_2")["Dual_SSM_state_dict"], strict=False)
Dual_SSM.eval()
closed_loop = build_closed_loop(Dual_SSM)


def denorm(x):
    return scaler_glucose.denormalize(x.detach()).numpy()


metrics = {"patient": patient}
with torch.no_grad():
    u0, u1, y, t = unpack(next(iter(train_loader)))
    y_np, y_hat_np = denorm(y), denorm(simulate_closed_loop(closed_loop, u0, u1, y, t)[0])
    metrics["FIT_train"] = float(np.median([FIT_formula(y_np[i, :, 0], y_hat_np[i, :, 0]) for i in range(y_np.shape[0])]))
    plot_glucose_insulin(t[0, :, 0].numpy(), meal=u0[0, :, 0].numpy(), insulin=u1[0, :, 0].numpy(),
                         glucose=y_np[0, :, 0], predicted_glucose=y_hat_np[0, :, 0], title="Training set, first sequence")

    u0, u1, y, t = unpack(next(iter(test_loader)))
    y_np, y_hat_np = denorm(y), denorm(simulate_closed_loop(closed_loop, u0, u1, y, t)[0])
    metrics["FIT_test"] = float(FIT_formula(y_np[0, :, 0], y_hat_np[0, :, 0]))
    plot_glucose_insulin(t[0, :, 0].numpy(), meal=u0[0, :, 0].numpy(), insulin=u1[0, :, 0].numpy(),
                         glucose=y_np[0, :, 0], predicted_glucose=y_hat_np[0, :, 0], title="Test set")

    for mode, path in (("insulin", impulse_insulin_path), ("meal", impulse_meal_path)):
        u0, u1, y, t = load_impulse_test(path, patient, scaler_meal, scaler_insulin, scaler_glucose, glucose_key="CGM")
        y_hat = simulate_closed_loop(closed_loop, u0, u1, y, t)[0]
        metrics[f"delta_G_{mode}"], metrics[f"delta_t_{mode}"] = impulse_response_metrics(
            y_hat, u1 if mode == "insulin" else u0, scaler_glucose, mode)
        plot_glucose_insulin(t[0, :, 0].numpy(), meal=u0[0, :, 0].numpy(), insulin=u1[0, :, 0].numpy(),
                             glucose=denorm(y)[0, :, 0], predicted_glucose=denorm(y_hat)[0, :, 0], title=f"Impulse test, {mode}")

save_metrics(results_folder / f"paz_{patient:03d}.csv", metrics)
print(metrics)